# 05 · H1 и H2 на постановке v3: смысл ответа относительно цели пользователя

Копия 03 на данных v3 (см. JOURNAL, 2026-10-19: на v2 целевого разрыва не нашлось). Ответ инструмента содержит все поля, а существенное — то, что ограничивает запрос пользователя (срок, количество, бюджет).

Без вмешательства. Отвечает на развилку проекта:

| H1 (probe) | H2 (поведение) | Что значит |
|---|---|---|
| ✔ | ошибается | **целевой разрыв**: различает, но не действует → steering осмыслен |
| ✔ | почти не ошибается | ошибок мало → менять постановку задач |
| ✘ | — | ответ не читается из активаций → steering вряд ли поможет; проверять промпт / явное состояние |

> **Протокол** (`docs/HYPOTHESES.md`, `AGENTS.md`): вектор строится только на train,
> подбор — только на val, test открывается **один раз** на замороженный метод.
> Пайплайн это проверяет сам — если ячейка упала с `TestLockError` или `StageOrderError`,
> это защита сработала, а не баг.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [ ]:
cfg = ExperimentConfig.make(
    name="h1h2-v3-qwen3-1.7b", hypothesis="H2", notebook="notebooks/05_h1_h2_v3.ipynb",
    notes="H1/H2 на v3: читает ли модель смысл ответа относительно цели пользователя.",
    overrides={"model.name": "Qwen/Qwen3-1.7B", "data.n_tasks": 160, "data.generator": "v3"},
)
exp = Experiment(cfg)
exp.data()
exp.estimate()

In [ ]:
exp.baseline()
exp.results_table("val")

### Где ошибается база: по типам пар, доменам, явности условия, позиции

In [ ]:
recs = pd.DataFrame(exp.state["records"]["val"]["baseline"] + exp.state["records"]["train"]["baseline"])
recs.groupby(["pair_type", "domain", "explicit"])["ok"].agg(["mean", "count"])
recs["need"] = [next(t for t in exp.state["tasks"] if t.task_id == i).notes.replace("v3 ", "") or "access" for i in recs.task_id]
recs.groupby(["pair_type", "need", "branch"])["ok"].agg(["mean", "count"])

### Аудит ошибок (H2): какие действия, в каких ветках и позициях

Разрыв H2 — это «различает, но не действует». Ошибка, где модель прочла ответ, но выбрала разумное действие вне A*, или лексический артефакт постановки — не он.

In [ ]:
display(recs.groupby(["pair_type", "domain", "branch"])["ok"].agg(["mean", "count"]))
display(recs.groupby(["pair_type", "position"])["ok"].agg(["mean", "count"]))
bad = recs[~recs.ok]
bad.groupby(["pair_type", "domain", "branch", "explicit", "action"]).size().rename("n").reset_index()

In [ ]:
exp.probe()
viz.probe_heatmap(exp.probe_table());

In [ ]:
for h, v in exp.verdicts().items():
    print(f"{h}: {v.label:16} {v.reason}")

In [ ]:
display(Markdown(exp.report(
    conclusion="""…""",
    next_step="""…""",
)))